# 状态管理  State
- typedict: 字典
- dataclass： 数据的类
- pydantic：基础模型basemodule，也当作统一的基础数据

**这三个都不是langgraph的，都是python自带，或者别人的数据类型包**

```python
class OverAllState(TypedDict):
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverAllState) -> OverAllState:
    pre_id = state["cur_id"] # 只有字典才能这样调用
    return {
        "logs": ["node_1 运行完毕"],
        "cur_id": pre_id + ", node_1"
    }
```

```python
@dataclass
class OverAllState:
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id  # 现在是数据类，不再能用字典的方式了
    return {  # 虽然返回的还是字典，  但是python支持字典和dataclass进行转换，不报错
        "logs": ["node_1 运行完毕"],
        "cur_id": pre_id + ", node_1"
    }

    # 替换成类的写法
    return OverAllState(
        logs=state.logs + ['node_1 运行完毕'],
        cur_id=pre_id + ", node_1"
    )
```

```python
class OverAllState(BaseModel):
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id  # 现在是数据类，不再能用字典的方式了
    return {  # 虽然返回的还是字典，  但是python支持字典和数据类进行转换，不报错
        "logs": ["node_1 运行完毕"],
        "cur_id": pre_id + ", node_1"
    }
```

## 3.1.1 TypedDict

In [7]:
# typedDict 上面讲了
# typedDict，  dict字典， python自带， 配value+key， 不用起名字，直接写字典， 底层typedDict对接字典， 之后可以像字典一样使用 (前面例子)

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from operator import add

class OverAllState(TypedDict):
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverAllState) -> OverAllState:
    pre_id = state["cur_id"]
    return {
        "logs": ["node_1 运行完毕"],
        "cur_id": pre_id + ", node_1"
    }

def node_2(state: OverAllState) -> OverAllState:
    pre_id = state["cur_id"] # state是一个字典，直接使用中括号+key，可以得到value值
    return {
        "logs": ["node_2 运行完毕"],
        "cur_id": pre_id + ", node_2"
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", END)

graph = builder.compile()

print(graph.invoke({"cur_id": "start"}))

{'logs': ['node_1 运行完毕', 'node_2 运行完毕'], 'cur_id': 'start, node_1, node_2'}


## 3.1.2 dataclass

In [8]:
# dataclass本事是注解，多了一些修改属性的方法
from langgraph.graph import StateGraph, START, END
from typing import Annotated
from dataclasses import dataclass
from operator import add


@dataclass # 更符合java类的感觉， 但依赖python的自动和dict转换
class OverAllState:
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id # 属性调用方式由['字段名']变为.字段名。
    return OverAllState(
        logs=state.logs + ['node_11 运行完毕'],
        cur_id=pre_id + ", node_1"
    )

def node_2(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id # 属性调用方式由['字段名']变为.字段名。
    return {
        "logs": ["node_2 运行完毕"],
        "cur_id": pre_id + ", node_2"
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)

builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", END)

graph = builder.compile()
print(graph.invoke({"cur_id": "start"}))
print(graph.invoke(OverAllState([], "start"))) # 两种都可以

{'logs': ['node_11 运行完毕', 'node_2 运行完毕'], 'cur_id': 'start, node_1, node_2'}
{'logs': ['node_11 运行完毕', 'node_2 运行完毕'], 'cur_id': 'start, node_1, node_2'}


## 3.1.3. Pydantic

In [6]:
# 和dataclass装饰器一样，直接使用.的方法是调用，但是是使用父类的方式

from langgraph.graph import StateGraph, START, END
from typing import Annotated
from pydantic import BaseModel
from operator import add

####
class OverAllState(BaseModel):
    logs: Annotated[list[str], add]
    cur_id: str
####

def node_1(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id  # 不一样
    return {
        "logs": ["node_1 运行完毕"],
        "cur_id": pre_id + ", node_1"
    }

def node_2(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return {
        "logs": ["node_2 运行完毕"],
        "cur_id": pre_id + ", node_2"
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", END)

graph = builder.compile()

print(graph.invoke({"cur_id": "start"}))

{'logs': ['node_1 运行完毕', 'node_2 运行完毕'], 'cur_id': 'start, node_1, node_2'}


### 3.1.4 校验行为
看下三种有没有不一样

langchain学习的时候：dict和dataclass不会对返回结果校验，原样返回用户，只有pydantic对格式要求严格
langchain想要校验，只能用pydantic

但langgraph定义状态的时候， 三个都要求字段名称一直，但是处理方式不一样

三种都会异常，不涉及功能区别

如果字段对不上，整个会忽略，不会对状态字段修改

#### 推荐用法
推荐typedDict方式
通过字典的key检查，直接调用，不要用类更新，当成字典更新比较好，如果用类，就是后面类覆盖前面的类，最好用字典更新

假如上面不return logs2， logs1也不会消失